# 11 · Model identification: BE or SC?

**What this shows.** Which of the two candidate strategies better explains each animal's trial-by-trial
updates — boundary estimation (BE: a boundary learnt from reward) or statistical categorisation (SC: a learnt
stimulus distribution). Both models are fitted to the expert sessions of notebook 10 and scored on held-out
data, with two independent methods (grid search, amortised SBI) and a consensus. On the real cohort first
(all mice); a synthetic cohort, where the truth is known, is the check that the methods recover it.

This is a description of the animals, not a factor in the opto analyses: with 9 opto animals and few SC calls,
splitting the opto effects by strategy would be anecdotal.

**How a call is made.** The manuscript's model selection, copied: the cross-validation is repeated over seeds,
and the model with the lower held-out error is called when a paired Wilcoxon over the repeats gives p < 0.05
(the manuscript used an ANOVA over 64 seeds). The repeats vary the simulation noise and the posterior samples,
not the data split, so a significant call means the call is stable under that noise.

**What produced it.** One `model_identification` run per cohort, on the cluster (see `START_HERE.md`,
"Model identification"); fetch it with `sc-fetch results model_identification/all-mice`. The synthetic check:

```bash
sc-make-synthetic-cohort --distribution uniform
RUN=$(sc-new-run --report model_identification --cohort synthetic_uniform)
bash slurm/submit.sh grid-search --cohort synthetic_uniform --distribution uniform --fit-target update_matrix --run-id $RUN
bash slurm/submit.sh sbi-condition --cohort synthetic_uniform --distribution uniform --run-id $RUN
```

then `sc-grid-search ... --gather` and `sc-consensus`. The run holds `grid_search/<fit_target>/<distribution>/`,
`sbi/...` and `consensus/<distribution>/assignments.csv`. Notebook 90 is where the SBI summary statistics were
chosen.

In [ ]:
from nb_setup import *
from sound_categorisation.inference.cv_utils import load_cv_results
from sound_categorisation.plotting.cv import plot_confusion, plot_cv_comparison, plot_winner_summary
from sound_categorisation.data.paths import model_id_dir
banner()
run = open_run('model_identification', MODEL_COHORT)
DIST, FT = 'uniform', 'update_matrix'
gs = load_cv_results(model_id_dir(run, 'grid_search', FT, DIST))
print(f'grid search: {gs.comparison.animal_id.nunique()} animals, {gs.long.seed.nunique()} seeds')

## 1 · Grid search: held-out error of each model per animal

For every animal both models are fitted by grid search with block-aware folds; the score is the held-out
MSE of the predicted update matrix. `winner` is the lower mean; `p_value` the paired Wilcoxon over seeds
(stability under simulation noise, as above).

In [ ]:
gs.comparison.round(4)

In [ ]:
aid = gs.comparison.animal_id.iloc[0]
plot_cv_comparison(gs.long, gs.comparison, aid, fit_target='UM');

## 2 · Recovery on the synthetic cohort

If `true_model` is in the comparison table (synthetic cohorts), the confusion matrix is the recovery
rate of this method at this cohort size. With the `--fast` settings CI uses this is a pipe-clean, not a
power estimate.

In [ ]:
if 'true_model' in gs.comparison and gs.comparison.true_model.notna().any():
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.4))
    plot_confusion(gs.comparison, ax=axes[0])
    axes[1].remove(); axes[1] = fig.add_subplot(1, 2, 2)
    w = plot_winner_summary(gs.comparison)
    plt.close(w)
    acc = gs.comparison.correct.mean()
    axes[1].bar(['correct', 'wrong'], [acc, 1 - acc], color=['0.4', '0.8']); axes[1].set(ylim=(0, 1), title=f'GS recovery {acc:.2f}')
    plt.tight_layout()
else:
    print('real cohort: no ground truth; see the consensus below')

## 3 · SBI, when present

The amortised networks (`sc-sbi-train`) are conditioned on each animal's expert data (`sc-sbi-condition`),
and the posterior-predictive held-out error is scored in the same schema, so `load_cv_results` reads it
unchanged. Absent in CI (no torch) — the cell says so rather than failing.

In [ ]:
sbi_dirs = sorted((run / 'sbi' / FT / DIST).glob('*')) if (run / 'sbi' / FT / DIST).exists() else []
if sbi_dirs:
    for d in sbi_dirs:
        r = load_cv_results(d)
        print(f'SBI rep {d.name}:'); display(r.comparison.round(4))
else:
    print('no SBI results in this run (sc-sbi-condition needs torch + trained networks)')

## 4 · Consensus

`sc-consensus` collects each method's significant call and applies the rule: fewer than `min_votes`
significant votes → Unclear; strict majority → that model; tie → Split. One row per animal.

In [ ]:
asg, summary = load_consensus(run, DIST)
print(summary)
cols = [c for c in asg.columns if not c.endswith('_p')]
asg[cols]

## 5 · Reading the real cohort

With `SC_NB_SYNTHETIC` off this notebook shows the all-mice run: §1 and §4 are the real animals and §2 is
skipped (no ground truth). An Unclear call means neither model was reliably better for that animal, not that
the animal uses neither strategy.